In [19]:
import re
import os
import glob
import math
from datetime import datetime
import pandas as pd
from datetime import datetime
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font

In [20]:
print("Current directory:", os.getcwd())

Current directory: /Users/oliverglanz/Library/CloudStorage/OneDrive-AndrewsUniversity/0000_EfficiencyWithIT/SmartSeminary/jupyter_notebooks


# Create DropDown Menues from a separate File

In [21]:
import pandas as pd
from openpyxl import load_workbook
from openpyxl.utils import get_column_letter, quote_sheetname
from openpyxl.worksheet.datavalidation import DataValidation

# ------------------------------------------------------------
# FILES
# ------------------------------------------------------------
DropDown_RETRIEVAL = "/Users/oliverglanz/Library/CloudStorage/OneDrive-AndrewsUniversity/0000_EfficiencyWithIT/SmartSeminary/0_source_files/0000_BuildingFiles/source_DropDownMenus_v20260505.xlsx"
target_file = "/Users/oliverglanz/Library/CloudStorage/OneDrive-AndrewsUniversity/0000_EfficiencyWithIT/SmartSeminary/0_source_files/default_DonSheet/DonSheet_default_empty_v20260506.xlsx"

dd_sheet_name = "DropDownMenu"
APPLY_TO_ROW = 5000

# ------------------------------------------------------------
# STEP 1: Read dropdown source workbook into dict (fast + clean)
# ------------------------------------------------------------
df_dd = pd.read_excel(DropDown_RETRIEVAL, dtype=str)

dropdown_lists = {
    col: df_dd[col].dropna().astype(str).str.strip().unique().tolist()
    for col in df_dd.columns
}

# ------------------------------------------------------------
# STEP 2: Load target workbook
# ------------------------------------------------------------
wb = load_workbook(target_file)

# ------------------------------------------------------------
# STEP 3: Create/refresh DropDownMenu sheet inside target workbook
# ------------------------------------------------------------
if dd_sheet_name in wb.sheetnames:
    ws_dd = wb[dd_sheet_name]
    wb.remove(ws_dd)  # remove entirely (faster + avoids leftover values)
ws_dd = wb.create_sheet(dd_sheet_name)

dropdown_ranges = {}
columns_with_false_default = set()

col_idx = 1
for header, values in dropdown_lists.items():
    if not values:
        continue

    ws_dd.cell(row=1, column=col_idx, value=header)

    # write values
    for row_idx, val in enumerate(values, start=2):
        ws_dd.cell(row=row_idx, column=col_idx, value=val)

    last_row = 1 + len(values)
    col_letter = get_column_letter(col_idx)

    dropdown_ranges[header] = f"{quote_sheetname(dd_sheet_name)}!${col_letter}$2:${col_letter}${last_row}"

    if "False" in values:
        columns_with_false_default.add(header)

    col_idx += 1

ws_dd.sheet_state = "hidden"

# ------------------------------------------------------------
# STEP 4: Apply ONLY dropdown validations to all sheets
# ------------------------------------------------------------
for ws in wb.worksheets:
    if ws.title == dd_sheet_name:
        continue

    # Map header -> column index for row 1 (fast lookups)
    headers = [cell.value for cell in ws[1]]
    header_to_idx = {h: i + 1 for i, h in enumerate(headers) if h}

    # Add data validations for headers that exist on the sheet
    for header, formula_range in dropdown_ranges.items():
        col_i = header_to_idx.get(header)
        if not col_i:
            continue

        col_letter = get_column_letter(col_i)

        dv = DataValidation(type="list", formula1=f"={formula_range}", allow_blank=True)
        ws.add_data_validation(dv)
        dv.add(f"${col_letter}$2:${col_letter}${APPLY_TO_ROW}")

        # Optional: set default "False" for blank cells for those dropdowns
        if header in columns_with_false_default:
            max_row = min(ws.max_row, APPLY_TO_ROW)
            for r in range(2, max_row + 1):
                cell = ws[f"{col_letter}{r}"]
                if cell.value is None or str(cell.value).strip() == "":
                    cell.value = "False"

# ------------------------------------------------------------
# STEP 5: Save
# ------------------------------------------------------------
wb.save(target_file)
wb.close()

print("✅ Applied dropdown menus only (no formatting changes).")


✅ Applied dropdown menus only (no formatting changes).


# Creating AutoFreeze, Autofilter

In [22]:
import warnings
from openpyxl import load_workbook
from openpyxl.styles import Alignment
from openpyxl.utils import get_column_letter

warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")

# ------------------------------------------------------------
# Input / Output
# ------------------------------------------------------------
input_file  = "/Users/oliverglanz/Library/CloudStorage/OneDrive-AndrewsUniversity/0000_EfficiencyWithIT/SmartSeminary/0_source_files/default_DonSheet/DonSheet_default_empty_v20260506.xlsx"
output_file = input_file

# ------------------------------------------------------------
# Load workbook
# ------------------------------------------------------------
wb = load_workbook(input_file)

def format_sheet(ws):
    # ------------------------------------------------------------
    # 1. Header formatting: vertical, bottom-center
    # ------------------------------------------------------------
    header_alignment = Alignment(
        textRotation=90,
        vertical="bottom",
        horizontal="center",
        wrap_text=False
    )

    for cell in ws[1]:
        cell.alignment = header_alignment

    # ------------------------------------------------------------
    # 2. Apply autofilter
    # ------------------------------------------------------------
    ws.auto_filter.ref = ws.dimensions

    # ------------------------------------------------------------
    # 3. Autofit column width (IGNORE header row)
    # ------------------------------------------------------------
    for col_idx in range(1, ws.max_column + 1):
        col_letter = get_column_letter(col_idx)
        max_length = 0

        for row in ws.iter_rows(min_row=2, min_col=col_idx, max_col=col_idx):
            val = row[0].value
            if val is None:
                continue

            text = str(val)

            # If there are line breaks, use the longest line
            if "\n" in text:
                text = max(text.split("\n"), key=len)

            max_length = max(max_length, len(text))

        ws.column_dimensions[col_letter].width = max(10, min(max_length + 2, 60))

# ------------------------------------------------------------
# Apply formatting to ALL sheets (skip hidden dropdown sheets if desired)
# ------------------------------------------------------------
SKIP_SHEETS = {"DropDown", "DropDownMenu"}  # adjust as needed

for ws in wb.worksheets:
    if ws.title in SKIP_SHEETS:
        continue
    format_sheet(ws)

# ------------------------------------------------------------
# Save to output
# ------------------------------------------------------------
wb.save(output_file)
wb.close()

print(f"✅ Saved formatted workbook to: {output_file}")


✅ Saved formatted workbook to: /Users/oliverglanz/Library/CloudStorage/OneDrive-AndrewsUniversity/0000_EfficiencyWithIT/SmartSeminary/0_source_files/default_DonSheet/DonSheet_default_empty_v20260506.xlsx


# Applying Formatting to the DropDownMenu

In [23]:
import os
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Border, Side, Alignment
from openpyxl.utils import get_column_letter, column_index_from_string
from openpyxl.utils.cell import quote_sheetname
from openpyxl.worksheet.datavalidation import DataValidation

INPUTFILE = "/Users/oliverglanz/Library/CloudStorage/OneDrive-AndrewsUniversity/0000_EfficiencyWithIT/SmartSeminary/0_source_files/0000_BuildingFiles/source_DropDownMenus_v20260505.xlsx"
wb = load_workbook(INPUTFILE)

# ------------------------------------------------------------
# 1) Rebuild dropdown ranges from hidden DropDownMenu sheet
# ------------------------------------------------------------
def build_dropdown_ranges_from_dd_sheet(wb, dd_sheet_name="DropDownMenu"):
    if dd_sheet_name not in wb.sheetnames:
        return {}, {}

    ws_dd = wb[dd_sheet_name]
    dropdown_ranges = {}
    dropdown_values = {}

    for col_idx in range(1, ws_dd.max_column + 1):
        header = ws_dd.cell(row=1, column=col_idx).value
        if header is None or str(header).strip() == "":
            continue

        values = []
        for r in range(2, ws_dd.max_row + 1):
            v = ws_dd.cell(row=r, column=col_idx).value
            if v is None or str(v).strip() == "":
                continue
            values.append(str(v))

        if not values:
            continue

        col_letter = get_column_letter(col_idx)
        last_row = 1 + len(values)

        dropdown_ranges[header] = f"{quote_sheetname(dd_sheet_name)}!${col_letter}$2:${col_letter}${last_row}"
        dropdown_values[header] = values

    return dropdown_ranges, dropdown_values


def clear_data_validations(sheet):
    sheet.data_validations.dataValidation = []


def reapply_dropdown_validations(sheet, dropdown_ranges, dropdown_values, max_rows=5000):
    headers = [cell.value for cell in sheet[1]]

    for header, formula_range in dropdown_ranges.items():
        if header not in headers:
            continue

        cidx = headers.index(header) + 1
        cL = get_column_letter(cidx)

        dv = DataValidation(type="list", formula1=f"={formula_range}", allow_blank=True)
        dv.showDropDown = False
        sheet.add_data_validation(dv)
        dv.add(f"${cL}$2:${cL}${max_rows}")

        values_list = dropdown_values.get(header, [])
        if "False" in values_list:
            for r in range(2, min(sheet.max_row or 1, max_rows) + 1):
                cell = sheet[f"{cL}{r}"]
                if cell.value is None or str(cell.value).strip() == "":
                    cell.value = "False"


dropdown_ranges, dropdown_values = build_dropdown_ranges_from_dd_sheet(wb, dd_sheet_name="DropDownMenu")

# ------------------------------------------------------------
# 2) Grouping
# ------------------------------------------------------------
GROUP_DEFS = [
    ("R", "AB"),
    ("AD", "AW"),
    ("AY", "BJ"),
    ("BK", "BV"),
    ("BX", "CE"),
    ("CG", "DQ")
]

def group_columns(sheet, group_defs=GROUP_DEFS, outline_level=1):
    # Show outline controls and place the summary column on the LEFT
    sheet.sheet_view.showOutlineSymbols = True
    sheet.sheet_properties.outlinePr.summaryRight = False
    sheet.sheet_properties.outlinePr.summaryBelow = True
    sheet.sheet_properties.outlinePr.applyStyles = True

    max_col = sheet.max_column or 1

    for start_col, end_col in group_defs:
        start_idx = column_index_from_string(start_col)
        end_idx = min(column_index_from_string(end_col), max_col)

        if start_idx >= end_idx or start_idx > max_col:
            continue

        # For a left-collapsing outline, Excel needs a summary column on the left.
        # So the first column stays visible, and the columns to its right are grouped.
        detail_start_idx = start_idx + 1

        if detail_start_idx <= end_idx:
            for col_idx in range(detail_start_idx, end_idx + 1):
                col_letter = get_column_letter(col_idx)
                dim = sheet.column_dimensions[col_letter]
                dim.outline_level = outline_level
                dim.hidden = False

            # Mark the left summary column so Excel keeps the collapse handle on the left
            summary_letter = get_column_letter(start_idx)
            sheet.column_dimensions[summary_letter].collapsed = False


# ------------------------------------------------------------
# 3) Force all data cells (rows 2+) to Text (strings)
# ------------------------------------------------------------
def force_all_columns_to_text(sheet, max_rows=5000):
    max_row = min(sheet.max_row or 1, max_rows)

    for row in sheet.iter_rows(min_row=2, max_row=max_row, min_col=1, max_col=sheet.max_column):
        for cell in row:
            if cell.value is None:
                continue
            cell.value = str(cell.value)
            cell.number_format = "@"

# ------------------------------------------------------------
# 4) Autofit widths
# ------------------------------------------------------------
def autofit_columns_ignore_header(sheet, min_width=4, max_width=60, padding=2, max_rows=5000):
    max_row = min(sheet.max_row or 1, max_rows)
    max_col = sheet.max_column or 1

    for col_idx in range(1, max_col + 1):
        max_length = 0
        for r in range(2, max_row + 1):
            v = sheet.cell(row=r, column=col_idx).value
            if v is None:
                continue
            max_length = max(max_length, len(str(v)))

        col_letter = get_column_letter(col_idx)
        sheet.column_dimensions[col_letter].width = max(min_width, min(max_length + padding, max_width))

# ------------------------------------------------------------
# 5) Fixed widths
# ------------------------------------------------------------
FIXED_WIDTHS = {
    "Notes {not in Banner}": 40,
    "Instructor Name {Instr Name}": 20,
    "Program {not in Banner}": 12,
    "Catalog Title": 25,
    "Section Title": 25,
    "Schedule Type {not in Banner}": 15,
    "Instruction Method {Inst Method}": 10,
    "Building {Meet Bldg}": 30,
    "Instructor Email {Instr Email}": 20,
    "Instructor ID {Instr ID}": 10,
    "load/contract {not in Banner}": 15,
    "Reason for Contract {not in Banner}": 15,
    "account to be charged {not in Banner}": 25,
}

def apply_fixed_widths(sheet, fixed_widths=FIXED_WIDTHS):
    headers = [cell.value for cell in sheet[1]]
    for header, width in fixed_widths.items():
        if header in headers:
            col_idx = headers.index(header) + 1
            sheet.column_dimensions[get_column_letter(col_idx)].width = width

# ------------------------------------------------------------
# 6) Formatting
# ------------------------------------------------------------
thin_grey_border = Border(
    left=Side(style="thin", color="D3D3D3"),
    right=Side(style="thin", color="D3D3D3"),
    top=Side(style="thin", color="D3D3D3"),
    bottom=Side(style="thin", color="D3D3D3"),
)

def format_sheet(sheet):
    headers = [cell.value for cell in sheet[1]]

    # Freeze panes
    if "Notes {not in Banner}" in headers:
        notes_idx = headers.index("Notes {not in Banner}") + 1
        sheet.freeze_panes = f"{get_column_letter(notes_idx + 1)}2"
    else:
        sheet.freeze_panes = "A2"

    # AutoFilter
    sheet.auto_filter.ref = None
    last_col = get_column_letter(sheet.max_column or 1)
    last_row = sheet.max_row or 1
    sheet.auto_filter.ref = f"A1:{last_col}{last_row}"

    # Header style
    for cell in sheet[1]:
        if cell.value is None or str(cell.value).strip() == "":
            continue
        cell.font = Font(
            name="Arial",
            size=10,
            bold=True,
            color="FF0000" if cell.value == "Notes {not in Banner}" else "000000",
        )
        cell.fill = PatternFill(start_color="D3D3D3", end_color="D3D3D3", fill_type="solid")
        cell.alignment = Alignment(
            horizontal="center",
            vertical="bottom",
            wrap_text=True,
            text_rotation=90
        )

    sheet.row_dimensions[1].height = 150

    # Light-blue header block
    start_h = "Don {not in Banner}"
    end_h = "SEM Department {Scacrse Dept}"
    if start_h in headers and end_h in headers:
        start_idx = headers.index(start_h) + 1
        end_idx = headers.index(end_h) + 1
        light_blue_fill = PatternFill(start_color="DDEBF7", end_color="DDEBF7", fill_type="solid")
        for col_idx in range(start_idx, end_idx + 1):
            sheet[f"{get_column_letter(col_idx)}1"].fill = light_blue_fill

    # Borders
    for row in sheet.iter_rows(min_row=2, max_row=sheet.max_row or 1, min_col=1, max_col=sheet.max_column or 1):
        for cell in row:
            cell.border = thin_grey_border

    # Fonts: global default Arial 10
    for row in sheet.iter_rows(min_row=1, max_row=sheet.max_row or 1, min_col=1, max_col=sheet.max_column or 1):
        for cell in row:
            f = cell.font or Font()
            cell.font = Font(
                name="Arial",
                size=10,
                bold=f.bold,
                italic=f.italic,
                underline=f.underline,
                color=f.color,
            )

    # Override columns A–G to Arial 12 (DATA ROWS ONLY)
    for row in sheet.iter_rows(min_row=2, max_row=sheet.max_row or 1, min_col=1, max_col=min(7, sheet.max_column or 1)):
        for cell in row:
            f = cell.font or Font()
            cell.font = Font(
                name="Arial",
                size=12,
                bold=f.bold,
                italic=f.italic,
                underline=f.underline,
                color=f.color,
            )
    # Blue header cells for selected columns
    blue_headers = {
        "pre-work # of weeks {not in Banner}",
        "pre-work hours/week {not in Banner}",
        "pre-work hours/period {not in Banner}",
        "intensive # of weeks {not in Banner}",
        "intensive hours/week {not in Banner}",
        "intensive hours/period {not in Banner}",
        "post-work # of weeks {not in Banner}",
        "post-work hours/week {not in Banner}",
        "post-work hours/period {not in Banner}",
        "sum of weeks (for check) {not in Banner}",
        "total contract hours {not in Banner}",
        "Year {not in Banner}",
        "Semester {not in Banner}",
        "Remote Employee {not in Banner}",
        "% Responsibility, load/contract {not in Banner}",
        "Reason for Contract {not in Banner}",
        "costs per credit {not in Banner}",
        "total costs {not in Banner}",
        "dept budget {not in Banner}",
        "account to be charged {not in Banner}",
        "SEM Department {Scacrse Dept}",
    }

    blue_fill = PatternFill(start_color="00B0F0", end_color="00B0F0", fill_type="solid")

    for col_idx, header in enumerate(headers, start=1):
        if header in blue_headers:
            sheet.cell(row=1, column=col_idx).fill = blue_fill


# ------------------------------------------------------------
# 7) Apply ONLY to sheet "DropDownMenu"
# ------------------------------------------------------------
TARGET_SHEET = "DropDownMenu"

if TARGET_SHEET not in wb.sheetnames:
    raise ValueError(f"Sheet '{TARGET_SHEET}' not found in workbook.")

ws = wb[TARGET_SHEET]

format_sheet(ws)
force_all_columns_to_text(ws, max_rows=5000)
group_columns(ws)
autofit_columns_ignore_header(ws)
apply_fixed_widths(ws)
clear_data_validations(ws)
reapply_dropdown_validations(ws, dropdown_ranges, dropdown_values, max_rows=5000)

# ------------------------------------------------------------
# 8) Safe save in place via temp file
# ------------------------------------------------------------
temp_file = INPUTFILE.replace(".xlsx", "_temp.xlsx")
wb.save(temp_file)
wb.close()

os.replace(temp_file, INPUTFILE)

print("✅ Workbook updated in place. Code applied only to sheet 'DropDownMenu'.")

✅ Workbook updated in place. Code applied only to sheet 'DropDownMenu'.


# Apply same formatting to the default DonSheet (modifications were needed)

In [24]:
import os
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Border, Side, Alignment
from openpyxl.utils import get_column_letter, column_index_from_string
from openpyxl.utils.cell import quote_sheetname
from openpyxl.worksheet.datavalidation import DataValidation

INPUTFILE = "/Users/oliverglanz/Library/CloudStorage/OneDrive-AndrewsUniversity/0000_EfficiencyWithIT/SmartSeminary/0_source_files/default_DonSheet/DonSheet_default_empty_v20260506.xlsx"
wb = load_workbook(INPUTFILE)

# ------------------------------------------------------------
# 1) Rebuild dropdown ranges from hidden DropDownMenu sheet
# ------------------------------------------------------------
def build_dropdown_ranges_from_dd_sheet(wb, dd_sheet_name="DropDownMenu"):
    if dd_sheet_name not in wb.sheetnames:
        return {}, {}

    ws_dd = wb[dd_sheet_name]
    dropdown_ranges = {}
    dropdown_values = {}

    for col_idx in range(1, ws_dd.max_column + 1):
        header = ws_dd.cell(row=1, column=col_idx).value
        if header is None or str(header).strip() == "":
            continue

        values = []
        for r in range(2, ws_dd.max_row + 1):
            v = ws_dd.cell(row=r, column=col_idx).value
            if v is None or str(v).strip() == "":
                continue
            values.append(str(v))

        if not values:
            continue

        col_letter = get_column_letter(col_idx)
        last_row = 1 + len(values)

        dropdown_ranges[header] = f"{quote_sheetname(dd_sheet_name)}!${col_letter}$2:${col_letter}${last_row}"
        dropdown_values[header] = values

    return dropdown_ranges, dropdown_values


def clear_data_validations(sheet):
    sheet.data_validations.dataValidation = []


def reapply_dropdown_validations(sheet, dropdown_ranges, dropdown_values, max_rows=5000):
    headers = [cell.value for cell in sheet[1]]

    for header, formula_range in dropdown_ranges.items():
        if header not in headers:
            continue

        cidx = headers.index(header) + 1
        cL = get_column_letter(cidx)

        dv = DataValidation(type="list", formula1=f"={formula_range}", allow_blank=True)
        dv.showDropDown = False
        sheet.add_data_validation(dv)
        dv.add(f"${cL}$2:${cL}${max_rows}")

        values_list = dropdown_values.get(header, [])
        if "False" in values_list:
            for r in range(2, min(sheet.max_row or 1, max_rows) + 1):
                cell = sheet[f"{cL}{r}"]
                if cell.value is None or str(cell.value).strip() == "":
                    cell.value = "False"


dropdown_ranges, dropdown_values = build_dropdown_ranges_from_dd_sheet(wb, dd_sheet_name="DropDownMenu")

# ------------------------------------------------------------
# 2) Grouping
# ------------------------------------------------------------
GROUP_DEFS = [
    ("Q", "AB"),
    ("AC", "AW"),
    ("AX", "BJ"),
    ("BK", "BV"),
    ("BW", "CE"),
    ("CF", "DQ")
]
def clear_existing_column_grouping(sheet):
    for col_letter, dim in sheet.column_dimensions.items():
        dim.outline_level = 0
        dim.hidden = False
        dim.collapsed = False

def group_columns(sheet, group_defs=GROUP_DEFS, outline_level=1):
    # Remove/override all existing column grouping first
    clear_existing_column_grouping(sheet)

    # Show outline controls and place the summary column on the LEFT
    sheet.sheet_view.showOutlineSymbols = True
    sheet.sheet_properties.outlinePr.summaryRight = False
    sheet.sheet_properties.outlinePr.summaryBelow = True
    sheet.sheet_properties.outlinePr.applyStyles = True

    max_col = sheet.max_column or 1

    for start_col, end_col in group_defs:
        start_idx = column_index_from_string(start_col)
        end_idx = min(column_index_from_string(end_col), max_col)

        if start_idx >= end_idx or start_idx > max_col:
            continue

        detail_start_idx = start_idx + 1

        if detail_start_idx <= end_idx:
            for col_idx in range(detail_start_idx, end_idx + 1):
                col_letter = get_column_letter(col_idx)
                dim = sheet.column_dimensions[col_letter]
                dim.outline_level = outline_level
                dim.hidden = False
                dim.collapsed = False

            summary_letter = get_column_letter(start_idx)
            sheet.column_dimensions[summary_letter].collapsed = False


# ------------------------------------------------------------
# 3) Force all data cells (rows 2+) to Text (strings)
# ------------------------------------------------------------
def force_all_columns_to_text(sheet, max_rows=5000):
    max_row = min(sheet.max_row or 1, max_rows)

    for row in sheet.iter_rows(min_row=2, max_row=max_row, min_col=1, max_col=sheet.max_column):
        for cell in row:
            if cell.value is None:
                continue
            cell.value = str(cell.value)
            cell.number_format = "@"

# ------------------------------------------------------------
# 4) Autofit widths
# ------------------------------------------------------------
def autofit_columns_ignore_header(sheet, min_width=4, max_width=60, padding=2, max_rows=5000):
    max_row = min(sheet.max_row or 1, max_rows)
    max_col = sheet.max_column or 1

    for col_idx in range(1, max_col + 1):
        max_length = 0
        for r in range(2, max_row + 1):
            v = sheet.cell(row=r, column=col_idx).value
            if v is None:
                continue
            max_length = max(max_length, len(str(v)))

        col_letter = get_column_letter(col_idx)
        sheet.column_dimensions[col_letter].width = max(min_width, min(max_length + padding, max_width))

# ------------------------------------------------------------
# 5) Fixed widths
# ------------------------------------------------------------
FIXED_WIDTHS = {
"Notes {not in Banner}": 40,
"Program {not in Banner}": 12,
"Catalog Title": 25,
"Section Title": 25,
"Schedule Type {not in Banner}": 15,
"Instruction Method {Inst Method}": 10,
"Meeting Type": 5,
"Semester Start Date {Soaterm Start Date}": 10,
"Pre-work Start Date {not in Banner}": 10,
"Pre-work End Date {not in Banner}": 10,
"Intensive Period Start Date {Meet Start Date}": 10,
"Intensive Period End Date {Meet End Date}": 10,
"Post-work Start Date {not in Banner}": 10,
"Post-work End Date {not in Banner}": 10,
"Semester End Date {Soaterm End Date}": 10,
"Course Beginning Time {Meet Beg Time}": 5,
"Course Ending Time {Meet End Time}": 5,
"Year {not in Banner}": 5,
"Semester {not in Banner}": 7,
"Room {Meet Room}": 6,
"Building {Meet Bldg}": 30,
"Instructor Name {Instr Name}": 20,
"Instructor Email {Instr Email}": 20,
"Instructor ID {Instr ID}": 10,
"load/contract {not in Banner}": 12,
"Reason for Contract {not in Banner}": 20,
"costs per credit {not in Banner}": 5,
"account to be charged {not in Banner}": 25,
"SEM Department {Scacrse Dept}": 5,
"dean email": 20,
"VP finance email": 20,
"HR email": 20,
}

def apply_fixed_widths(sheet, fixed_widths=FIXED_WIDTHS):
    headers = [cell.value for cell in sheet[1]]
    for header, width in fixed_widths.items():
        if header in headers:
            col_idx = headers.index(header) + 1
            sheet.column_dimensions[get_column_letter(col_idx)].width = width

# ------------------------------------------------------------
# 6) Formatting
# ------------------------------------------------------------
thin_grey_border = Border(
    left=Side(style="thin", color="D3D3D3"),
    right=Side(style="thin", color="D3D3D3"),
    top=Side(style="thin", color="D3D3D3"),
    bottom=Side(style="thin", color="D3D3D3"),
)

def format_sheet(sheet):
    headers = [cell.value for cell in sheet[1]]

    # Freeze panes
    if "Notes {not in Banner}" in headers:
        notes_idx = headers.index("Notes {not in Banner}") + 1
        sheet.freeze_panes = f"{get_column_letter(notes_idx + 1)}2"
    else:
        sheet.freeze_panes = "A2"

    # AutoFilter
    sheet.auto_filter.ref = None
    last_col = get_column_letter(sheet.max_column or 1)
    last_row = sheet.max_row or 1
    sheet.auto_filter.ref = f"A1:{last_col}{last_row}"

    # Header style
    for cell in sheet[1]:
        if cell.value is None or str(cell.value).strip() == "":
            continue
        cell.font = Font(
            name="Arial",
            size=10,
            bold=True,
            color="FF0000" if cell.value == "Notes {not in Banner}" else "000000",
        )
        cell.fill = PatternFill(start_color="D3D3D3", end_color="D3D3D3", fill_type="solid")
        cell.alignment = Alignment(
            horizontal="center",
            vertical="bottom",
            wrap_text=True,
            text_rotation=90
        )

    sheet.row_dimensions[1].height = 150

    # Light-blue header block
    start_h = "Don {not in Banner}"
    end_h = "SEM Department {Scacrse Dept}"
    if start_h in headers and end_h in headers:
        start_idx = headers.index(start_h) + 1
        end_idx = headers.index(end_h) + 1
        light_blue_fill = PatternFill(start_color="DDEBF7", end_color="DDEBF7", fill_type="solid")
        for col_idx in range(start_idx, end_idx + 1):
            sheet[f"{get_column_letter(col_idx)}1"].fill = light_blue_fill

    # Borders
    for row in sheet.iter_rows(min_row=2, max_row=sheet.max_row or 1, min_col=1, max_col=sheet.max_column or 1):
        for cell in row:
            cell.border = thin_grey_border

    # Fonts: global default Arial 10
    for row in sheet.iter_rows(min_row=1, max_row=sheet.max_row or 1, min_col=1, max_col=sheet.max_column or 1):
        for cell in row:
            f = cell.font or Font()
            cell.font = Font(
                name="Arial",
                size=10,
                bold=f.bold,
                italic=f.italic,
                underline=f.underline,
                color=f.color,
            )

    # Override columns A–G to Arial 12 (DATA ROWS ONLY)
    for row in sheet.iter_rows(min_row=2, max_row=sheet.max_row or 1, min_col=1, max_col=min(7, sheet.max_column or 1)):
        for cell in row:
            f = cell.font or Font()
            cell.font = Font(
                name="Arial",
                size=12,
                bold=f.bold,
                italic=f.italic,
                underline=f.underline,
                color=f.color,
            )
    # Blue header cells for selected columns
    blue_headers = {
        "pre-work # of weeks {not in Banner}",
        "pre-work hours/week {not in Banner}",
        "pre-work hours/period {not in Banner}",
        "intensive # of weeks {not in Banner}",
        "intensive hours/week {not in Banner}",
        "intensive hours/period {not in Banner}",
        "post-work # of weeks {not in Banner}",
        "post-work hours/week {not in Banner}",
        "post-work hours/period {not in Banner}",
        "sum of weeks (for check) {not in Banner}",
        "total contract hours {not in Banner}",
        "Year {not in Banner}",
        "Semester {not in Banner}",
        "Remote Employee {not in Banner}",
        "% Responsibility, load/contract {not in Banner}",
        "Reason for Contract {not in Banner}",
        "costs per credit {not in Banner}",
        "total costs {not in Banner}",
        "dept budget {not in Banner}",
        "account to be charged {not in Banner}",
        "SEM Department {Scacrse Dept}",
    }

    blue_fill = PatternFill(start_color="00B0F0", end_color="00B0F0", fill_type="solid")

    for col_idx, header in enumerate(headers, start=1):
        if header in blue_headers:
            sheet.cell(row=1, column=col_idx).fill = blue_fill


# ------------------------------------------------------------
# 7) Apply to ALL sheets (except DropDownMenu)
# ------------------------------------------------------------
for sheet_name in wb.sheetnames:
    if sheet_name == "DropDownMenu":
        continue  # skip the dropdown source sheet

    ws = wb[sheet_name]

    format_sheet(ws)
    force_all_columns_to_text(ws, max_rows=5000)
    group_columns(ws)
    autofit_columns_ignore_header(ws)
    apply_fixed_widths(ws)
    clear_data_validations(ws)
    reapply_dropdown_validations(ws, dropdown_ranges, dropdown_values, max_rows=5000)

# ------------------------------------------------------------
# 8) Safe save in place via temp file
# ------------------------------------------------------------
temp_file = INPUTFILE.replace(".xlsx", "_temp.xlsx")
wb.save(temp_file)
wb.close()

os.replace(temp_file, INPUTFILE)

print("✅ Workbook updated in place. Code applied only to sheet 'DropDownMenu'.")

✅ Workbook updated in place. Code applied only to sheet 'DropDownMenu'.


# Adding Smart Formulas to the DonSheet

This cell imports the AutoFill source files into hidden helper sheets and applies formulas across every DonSheet tab. Run this after the dropdown/formatting cells so formulas are not converted to plain text.

In [25]:
from openpyxl import load_workbook
from openpyxl.utils import get_column_letter
import os

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------
MAIN_FILE = "../0_source_files/default_DonSheet/DonSheet_default_empty_v20260506.xlsx"

NAMES_FILE = "../0_source_files/0000_BuildingFiles/source_AutoFill_names_v20260505.xlsx"
COURSES_FILE = "../0_source_files/0000_BuildingFiles/source_AutoFill_courses_v20260505.xlsx"

HELPER_SHEET_NAMES = "_names_lookup"
HELPER_SHEET_COURSES = "_courses_lookup"

START_ROW = 2
END_ROW = 300

print(os.path.exists(MAIN_FILE))
print(os.path.exists(NAMES_FILE))
print(os.path.exists(COURSES_FILE))


# ------------------------------------------------------------
# LOAD WORKBOOK
# ------------------------------------------------------------
wb = load_workbook(MAIN_FILE)


# ------------------------------------------------------------
# LOAD SOURCE FILES INTO HIDDEN SHEETS
# ------------------------------------------------------------
def load_lookup_sheet(wb, filename, target_sheet_name):
    src_wb = load_workbook(filename)
    src_ws = src_wb.active

    if target_sheet_name in wb.sheetnames:
        del wb[target_sheet_name]

    tgt_ws = wb.create_sheet(title=target_sheet_name)

    for row in src_ws.iter_rows(values_only=True):
        tgt_ws.append(row)

    tgt_ws.sheet_state = "hidden"


load_lookup_sheet(wb, NAMES_FILE, HELPER_SHEET_NAMES)
load_lookup_sheet(wb, COURSES_FILE, HELPER_SHEET_COURSES)


# ------------------------------------------------------------
# ADD COURSE LOOKUP KEY: Subject + Course Number
# Assumes:
# A = Subject
# B = Course Number
# C = Catalog Title
# D = Section Title
# ------------------------------------------------------------
course_ws = wb[HELPER_SHEET_COURSES]
course_ws["E1"] = "Lookup Key"

for r in range(2, course_ws.max_row + 1):
    subject = course_ws[f"A{r}"].value
    course_num = course_ws[f"B{r}"].value

    if subject is not None and course_num is not None:
        course_ws[f"E{r}"] = f"{str(subject).strip()}{str(course_num).strip()}"


# ------------------------------------------------------------
# FIND COLUMN INDEX BY HEADER
# ------------------------------------------------------------
def get_col_index(sheet, header_name):
    headers = [cell.value for cell in sheet[1]]
    if header_name not in headers:
        return None
    return headers.index(header_name) + 1


# ------------------------------------------------------------
# APPLY FORMULAS TO EACH SHEET
# ------------------------------------------------------------
for sheet_name in wb.sheetnames:

    if sheet_name.startswith("_"):
        continue

    ws = wb[sheet_name]

    # ---- Column mappings ----
    col_instr = get_col_index(ws, "Instructor Name {Instr Name}")
    col_email = get_col_index(ws, "Instructor Email {Instr Email}")
    col_id = get_col_index(ws, "Instructor ID {Instr ID}")

    col_subject = get_col_index(ws, "Subject")
    col_course = get_col_index(ws, "Course Number {Crse Num}")
    col_catalog = get_col_index(ws, "Catalog Title")
    col_section = get_col_index(ws, "Section Title")

    # Dates
    col_pre_start = get_col_index(ws, "Pre-work Start Date {not in Banner}")
    col_pre_end = get_col_index(ws, "Pre-work End Date {not in Banner}")
    col_pre_weeks = get_col_index(ws, "pre-work # of weeks {not in Banner}")

    col_int_start = get_col_index(ws, "Intensive Period Start Date {Meet Start Date}")
    col_int_end = get_col_index(ws, "Intensive Period End Date {Meet End Date}")
    col_int_weeks = get_col_index(ws, "intensive # of weeks {not in Banner}")

    col_post_start = get_col_index(ws, "Post-work Start Date {not in Banner}")
    col_post_end = get_col_index(ws, "Post-work End Date {not in Banner}")
    col_post_weeks = get_col_index(ws, "post-work # of weeks {not in Banner}")

    col_sem_start = get_col_index(ws, "Semester Start Date {Soaterm Start Date}")
    col_fy = get_col_index(ws, "Fiscal Year {not in Banner}")
    col_sem = get_col_index(ws, "Semester {not in Banner}")

    for row in range(START_ROW, END_ROW + 1):

        # ----------------------------------------------------
        # 1) Instructor lookup
        # Names helper sheet:
        # A = Faculty Name
        # B = ID#
        # C = email
        # ----------------------------------------------------
        if col_instr and col_email:
            ws.cell(row=row, column=col_email).value = (
                f'=IFERROR(INDEX(\'{HELPER_SHEET_NAMES}\'!$C:$C,'
                f'MATCH({get_column_letter(col_instr)}{row},'
                f'\'{HELPER_SHEET_NAMES}\'!$A:$A,0)),"")'
            )

        if col_instr and col_id:
            ws.cell(row=row, column=col_id).value = (
                f'=IFERROR(INDEX(\'{HELPER_SHEET_NAMES}\'!$B:$B,'
                f'MATCH({get_column_letter(col_instr)}{row},'
                f'\'{HELPER_SHEET_NAMES}\'!$A:$A,0)),"")'
            )

        # ----------------------------------------------------
        # 2) Course lookup
        # Course helper sheet:
        # A = Subject
        # B = Course Number
        # C = Catalog Title
        # D = Section Title
        # E = Lookup Key
        # ----------------------------------------------------
        if col_subject and col_course and col_catalog:
            ws.cell(row=row, column=col_catalog).value = (
                f'=IFERROR(INDEX(\'{HELPER_SHEET_COURSES}\'!$C:$C,'
                f'MATCH({get_column_letter(col_subject)}{row}&{get_column_letter(col_course)}{row},'
                f'\'{HELPER_SHEET_COURSES}\'!$E:$E,0)),"")'
            )

        if col_subject and col_course and col_section:
            ws.cell(row=row, column=col_section).value = (
                f'=IFERROR(INDEX(\'{HELPER_SHEET_COURSES}\'!$D:$D,'
                f'MATCH({get_column_letter(col_subject)}{row}&{get_column_letter(col_course)}{row},'
                f'\'{HELPER_SHEET_COURSES}\'!$E:$E,0)),"")'
            )

        # ----------------------------------------------------
        # 3) Week calculations
        # ----------------------------------------------------
        def week_formula(start_col, end_col):
            return (
                f'=IF(AND({start_col}{row}<>"",{end_col}{row}<>""),'
                f'ROUND(({end_col}{row}-{start_col}{row})/7,1),"")'
            )

        if col_pre_start and col_pre_end and col_pre_weeks:
            ws.cell(row=row, column=col_pre_weeks).value = week_formula(
                get_column_letter(col_pre_start),
                get_column_letter(col_pre_end)
            )

        if col_int_start and col_int_end and col_int_weeks:
            ws.cell(row=row, column=col_int_weeks).value = week_formula(
                get_column_letter(col_int_start),
                get_column_letter(col_int_end)
            )

        if col_post_start and col_post_end and col_post_weeks:
            ws.cell(row=row, column=col_post_weeks).value = week_formula(
                get_column_letter(col_post_start),
                get_column_letter(col_post_end)
            )

        # ----------------------------------------------------
        # 4) Fiscal Year + Semester
        # ----------------------------------------------------
        if col_sem_start and col_fy:
            ws.cell(row=row, column=col_fy).value = (
                f'=IF({get_column_letter(col_sem_start)}{row}="","",'
                f'YEAR({get_column_letter(col_sem_start)}{row})+'
                f'IF(MONTH({get_column_letter(col_sem_start)}{row})>=5,1,0))'
            )

        if col_sem_start and col_sem:
            sem_cell = f"{get_column_letter(col_sem_start)}{row}"
        
            ws.cell(row=row, column=col_sem).value = (
                f'=IF({sem_cell}="","",'
                f'IF(AND(MONTH({sem_cell})=8,DAY({sem_cell})>=15),"Fall",'
                f'IF(MONTH({sem_cell})>8,"Fall",'
                f'IF(MONTH({sem_cell})=5,"Summer",'
                f'IF(MONTH({sem_cell})=1,"Spring","")))))'
            )


# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------
OUTPUT_FILE = "../0_source_files/default_DonSheet/DonSheet_default_empty_v20260506_formulas.xlsx"
wb.save(OUTPUT_FILE)

print(f"Saved: {OUTPUT_FILE}")

True
True
True
Saved: ../0_source_files/default_DonSheet/DonSheet_default_empty_v20260506_formulas.xlsx
